In [1]:
import tensorflow as tf

batch_size = 32
img_height = 224
img_width = 224

# Diviser les données en 80% train et 20% pour validation/test
train_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds) - val_size  # Reste pour test

val_ds = val_test_ds.take(val_size)  # Premier 50% pour validation
test_ds = val_test_ds.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds)}")
print(f"Nombre de batches dans val_ds: {len(val_ds)}")
print(f"Nombre de batches dans test_ds: {len(test_ds)}")

Found 1940 files belonging to 3 classes.
Using 1552 files for training.
Found 1940 files belonging to 3 classes.
Using 388 files for validation.
Nombre de batches dans train_ds: 49
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [5]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds_maiis))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds_maiis) - val_size  # Reste pour test

val_ds_maiis = val_test_ds_maiis.take(val_size)  # Premier 50% pour validation
test_ds_maiis = val_test_ds_maiis.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_maiis)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_maiis)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_maiis)}")

Found 1930 files belonging to 3 classes.
Using 1544 files for training.
Found 1930 files belonging to 3 classes.
Using 386 files for validation.
Nombre de batches dans train_ds: 49
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [6]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size_mixte = int(0.5 * len(val_test_ds_mixte))  # 50% de val_test_ds pour validation
test_size_mixte = len(val_test_ds_mixte) - val_size_mixte  # Reste pour test

val_ds_mixte = val_test_ds_maiis.take(val_size_mixte)  # Premier 50% pour validation
test_ds_mixte = val_test_ds_maiis.skip(val_size_mixte)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_mixte)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_mixte)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_mixte)}")

Found 1968 files belonging to 3 classes.
Using 1575 files for training.
Found 1968 files belonging to 3 classes.
Using 393 files for validation.
Nombre de batches dans train_ds: 50
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [2]:
from tensorflow.keras.applications import MobileNetV3Large  # Utiliser MobileNetV3Large ou MobileNetV3Small
from tensorflow.keras import layers, models
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import time

# Définir les dimensions d'image pour MobileNetV3
img_height, img_width = 224, 224

# Charger le modèle MobileNetV3 pré-entraîné
base_model = MobileNetV3Large(input_shape=(img_height, img_width, 3),
                              include_top=False,
                              weights='imagenet')
base_model.trainable = False  # Geler les poids du modèle pré-entraîné

for layer in base_model.layers[-50:]:
    layer.trainable = True

# Ajouter une couche Global Average Pooling
global_average_layer = layers.GlobalAveragePooling2D()(base_model.output)

# Ajouter des couches fully connected supplémentaires avec Dropout
dense_1 = layers.Dense(1024, activation='relu')(global_average_layer)
dropout_1 = layers.Dropout(0.5)(dense_1)

dense_2 = layers.Dense(512, activation='relu')(dropout_1)
dropout_2 = layers.Dropout(0.5)(dense_2)

dense_3 = layers.Dense(256, activation='relu')(dropout_2)
dropout_3 = layers.Dropout(0.5)(dense_3)

dense_4 = layers.Dense(128, activation='relu')(dropout_3)
dropout_4 = layers.Dropout(0.5)(dense_4)

# Créer les sorties pour chaque nutriment (13 au total)
outputs = []
for nutrient in range(13):
    output = layers.Dense(3, activation='softmax', name=f'nutrient_{nutrient}')(dropout_4)
    outputs.append(output)

# Créer le modèle final avec MobileNetV3 en entrée et les 13 sorties en sortie
model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilation du modèle avec des métriques adaptées pour chaque sortie
metrics = ['accuracy', Precision(name='precision'), Recall(name='recall')]
metrics_list = [metrics] * 13  # Appliquer les métriques à chaque nutriment

model.compile(optimizer='adam',
              loss=['categorical_crossentropy'] * 13,
              metrics=metrics_list)

# Afficher un résumé du modèle pour vérifier les couches et les sorties
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling (Rescaling)         │ (None, 224, 224, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv (Conv2D)                 │ (None, 112, 112, 16)      │             432 │ rescaling[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv_bn (BatchNormalization)  │ (None, 112, 112, 16)      │              64 │ conv[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 112, 112, 16)      │               0 │ conv_bn[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 16)      │             144 │ activation[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_bn    │ (None, 112, 112, 16)      │              64 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu (ReLU)                  │ (None, 112, 112, 16)      │               0 │ expanded_conv_depthwise_b… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             256 │ re_lu[0][0]                │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_bn      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_add (Add)       │ (None, 112, 112, 16)      │               0 │ activation[0][0],          │
│                               │                           │                 │ expanded_conv_project_bn[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_1_expand        │ (None, 112, 112, 64)      │           1,024 │ expanded_conv_add[0][0]    │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_1_expand_bn     │ (None, 112, 112, 64)      │             256 │ expanded_conv_1_expand[0]… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_1 (ReLU)                │ (None, 112, 112, 64)      │               

 Total params: 4,674,471 (17.83 MB)

 Trainable params: 3,857,807 (14.72 MB)

 Non-trainable params: 816,664 (3.12 MB)

In [3]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Définir le checkpoint pour sauvegarder les meilleurs poids
checkpoint = ModelCheckpoint('MobileNet_V3_weights.keras',
                             monitor='val_accuracy',
                             verbose=1,
                             mode='max',
                             save_best_only=True)

# Early stopping pour arrêter l'entraînement si la validation stagne
early = EarlyStopping(monitor="val_loss",
                      mode="min",
                      restore_best_weights=True,
                      patience=5)

# Liste des callbacks
callbacks_list = [checkpoint, early]

In [4]:
import time
# Entraînement du modèle tout en mesurant le temps
start_time = time.time()

history = model.fit(
    train_val_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks_list,
    verbose=True,
    shuffle=True
)

end_time = time.time()

# Afficher le temps d'entraînement
training_time = end_time - start_time
print(f"Temps d'apprentissage : {training_time} secondes")

# Calcul manuel du F1-score après l'entraînement
precision = history.history['precision'][-1]
recall = history.history['recall'][-1]
f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon()) 
print(f'F1 Score: {f1:.4f}')

Epoch 1/15


C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\optimizers\base_optimizer.py:678: UserWarning: Gradients do not exist for variables ['kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - loss: 1.2823 - nutrient_0_accuracy: 0.4146 - nutrient_0_precision: 0.4483 - nutrient_0_recall: 0.2939

C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\callbacks\model_checkpoint.py:206: UserWarning: Can save best model only with val_accuracy available, skipping.
  self._save_model(epoch=epoch, batch=None, logs=logs)


49/49 ━━━━━━━━━━━━━━━━━━━━ 408s 5s/step - loss: 1.2773 - nutrient_0_accuracy: 0.4164 - nutrient_0_precision: 0.4506 - nutrient_0_recall: 0.2954 - val_loss: 0.9331 - val_nutrient_0_accuracy: 0.6771 - val_nutrient_0_precision: 0.6771 - val_nutrient_0_recall: 0.6771
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 183s 4s/step - loss: 0.5753 - nutrient_0_accuracy: 0.6684 - nutrient_0_precision: 0.6998 - nutrient_0_recall: 0.6169 - val_loss: 3.4430 - val_nutrient_0_accuracy: 0.6771 - val_nutrient_0_precision: 0.6771 - val_nutrient_0_recall: 0.6771
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 173s 4s/step - loss: 0.5345 - nutrient_0_accuracy: 0.6990 - nutrient_0_precision: 0.7132 - nutrient_0_recall: 0.6594 - val_loss: 6.9468 - val_nutrient_0_accuracy: 0.5729 - val_nutrient_0_precision: 0.5759 - val_nutrient_0_recall: 0.5729
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 97s 2s/step - loss: 0.4398 - nutrient_0_accuracy: 0.7680 - nutrient_0_precision: 0.7838 - nutrient_0_recall: 0.7420 - val_loss: 8.9436 - val_nutrien

KeyError: 'precision'

In [7]:
model.save("model/MobileNetV3_04_11.h5")

In [8]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 300s 12s/step - loss: 0.8542 - nutrient_0_accuracy: 0.7139 - nutrient_0_precision: 0.7139 - nutrient_0_recall: 0.7139
Résultats de l'évaluation : [0.8783722519874573, 0.6938775777816772, 0.6938775777816772, 0.6938775777816772]


In [9]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 37s 3s/step - loss: 0.9327 - nutrient_0_accuracy: 0.6611 - nutrient_0_precision: 0.6611 - nutrient_0_recall: 0.6611
Nombre total de résultats: 4
Résultats de l'évaluation: [0.8687166571617126, 0.6632652878761292, 0.6632652878761292, 0.6632652878761292]
La structure des résultats est différente de celle attendue.


In [10]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_mixte)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 67s 3s/step - loss: 2.2146 - nutrient_0_accuracy: 0.4576 - nutrient_0_precision: 0.4548 - nutrient_0_recall: 0.4359
Résultats de l'évaluation : [2.1636743545532227, 0.45876288414001465, 0.4677419364452362, 0.4484536051750183]


In [11]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_mixte)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - loss: 2.0536 - nutrient_0_accuracy: 0.4373 - nutrient_0_precision: 0.4411 - nutrient_0_recall: 0.4195
Nombre total de résultats: 4
Résultats de l'évaluation: [1.9757083654403687, 0.438144326210022, 0.4462365508079529, 0.42783504724502563]
La structure des résultats est différente de celle attendue.


In [12]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_maiis)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - loss: 2.4748 - nutrient_0_accuracy: 0.3464 - nutrient_0_precision: 0.3589 - nutrient_0_recall: 0.3464
Résultats de l'évaluation : [2.246450662612915, 0.37628865242004395, 0.3903743326663971, 0.37628865242004395]


In [13]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_maiis)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - loss: 2.0601 - nutrient_0_accuracy: 0.4568 - nutrient_0_precision: 0.4622 - nutrient_0_recall: 0.4460
Nombre total de résultats: 4
Résultats de l'évaluation: [2.19771409034729, 0.42268040776252747, 0.4331550896167755, 0.4175257682800293]
La structure des résultats est différente de celle attendue.


In [14]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

6/6 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - loss: 0.8166 - nutrient_0_accuracy: 0.7021 - nutrient_0_precision: 0.7021 - nutrient_0_recall: 0.7021
Nombre total de résultats: 4
Résultats de l'évaluation: [0.7746589183807373, 0.703125, 0.703125, 0.703125]
La structure des résultats est différente de celle attendue.


In [15]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - loss: 0.8003 - nutrient_0_accuracy: 0.6730 - nutrient_0_precision: 0.6730 - nutrient_0_recall: 0.6730
Nombre total de résultats: 4
Résultats de l'évaluation: [0.9304707646369934, 0.6770833134651184, 0.6770833134651184, 0.6770833134651184]
La structure des résultats est différente de celle attendue.


In [16]:
import time
from tensorflow.keras import backend as K

# Fonction pour calculer la moyenne d'une métrique
def mean_metric(metric_values):
    return sum(metric_values) / len(metric_values)

# Nombre de nutriments
num_nutrients = 13

# Fonction pour calculer les moyennes pour chaque nutriment
def mean_nutrient_metric(metric_name, history):
    metrics = []
    for nutrient in range(num_nutrients):
        key = f'nutrient_{nutrient}_{metric_name}'
        if key in history:
            metrics.append(history[key])
    return [mean_metric(metric) for metric in zip(*metrics)]  # Moyenne sur les époques

# Calcul des moyennes pour l'ensemble des époques (entraînement)
mean_train_loss = mean_metric(history.history['loss'])
mean_train_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_train_precision = mean_nutrient_metric('precision', history.history)
mean_train_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (entraînement)
f1_train_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                   for p, r in zip(mean_train_precision, mean_train_recall)]
mean_f1_train = mean_metric(f1_train_scores)

# Calcul des moyennes pour l'ensemble des époques (validation)
mean_val_loss = mean_metric(history.history['val_loss'])
mean_val_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_val_precision = mean_nutrient_metric('precision', history.history)
mean_val_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (validation)
f1_val_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                 for p, r in zip(mean_val_precision, mean_val_recall)]
mean_f1_val = mean_metric(f1_val_scores)

# Affichage des résultats
print(f"Moyenne de la perte sur l'ensemble d'entraînement : {mean_train_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble d'entraînement : {mean_metric(mean_train_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble d'entraînement : {mean_metric(mean_train_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble d'entraînement : {mean_metric(mean_train_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble d'entraînement : {mean_f1_train:.4f}")

print(f"Moyenne de la perte sur l'ensemble de validation : {mean_val_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble de validation : {mean_metric(mean_val_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble de validation : {mean_metric(mean_val_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble de validation : {mean_metric(mean_val_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble de validation : {mean_f1_val:.4f}")

Moyenne de la perte sur l'ensemble d'entraînement : 0.5656
Moyenne de l'accuracy sur l'ensemble d'entraînement : 0.7079
Moyenne de la précision sur l'ensemble d'entraînement : 0.7286
Moyenne du rappel sur l'ensemble d'entraînement : 0.6633
Moyenne du F1-score sur l'ensemble d'entraînement : 0.6920
Moyenne de la perte sur l'ensemble de validation : 8.2637
Moyenne de l'accuracy sur l'ensemble de validation : 0.7079
Moyenne de la précision sur l'ensemble de validation : 0.7286
Moyenne du rappel sur l'ensemble de validation : 0.6633
Moyenne du F1-score sur l'ensemble de validation : 0.6920
